# 论文 24：机器超级智能

**Shane Legg 的博士论文 (2008)：通用人工智能**

通过可计算近似对通用智能、AIXI 智能体与 Solomonoff 归纳进行实践探索。

---

## 概述

该笔记本实现了 Shane Legg 在通用人工智能方面的基础工作中的关键概念：

- **通用智能**：智能的正式数学定义
- **AIXI 智能体**：使用 Solomonoff 归纳的最优强化学习智能体
- **Solomonoff 归纳**：序列预测的通用先验
- **Kolmogorov 复杂度**：测量信息内容
- **Monte Carlo AIXI**：使用采样的实际近似
- **智能度量**：量化智能体在不同环境中的表现

由于精确的 AIXI 不可计算，因此我们专注于使用玩具环境的**实用近似**。

---

## 内容

1. **智能理论** - 心理测量模型和 g 因子
2. **通用人工智能与 Solomonoff 归纳** - 序列预测和压缩
3. **AIXI 智能体与环境模型** - 玩具 MDP 中的 MC-AIXI
4. **通用智能度量** - 不同智能体的 Υ(π)
5. **近似方法与计算限制** - 有时间限制的 AIXI
6. **通往超级智能的路径** - 递归自我改进与智能爆炸

---

**注意**：此笔记本使用仅 NumPy 的实现以及用于教育目的的合成环境。运行时间控制在 5 分钟以内。

**相关内容**：
- 论文 23 (MDL)：模型选择的最小描述长度
- 论文 25（Kolmogorov复杂度）：信息理论基础
- 论文 8 (DQN)：实用深度 RL 与理论最优智能体

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, deque
from typing import List, Tuple, Dict, Optional
import itertools
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子以实现可重复性
np.random.seed(42)

# 绘图配置
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Imports complete!")
print("NumPy version:", np.__version__)

---

# 第 1 节：智能理论

## 1.1 心理测量智力

人类智能研究使用**心理测量模型**来测量认知能力。 **g 因子**（一般智力）源自不同认知测试之间的相关性。

### 斯皮尔曼的 g 因子

当人们进行多项认知测试（语言、空间、记忆、推理）时，测试中的表现呈正相关。这表明了一个潜在的共同因素：**一般智力（g）**。

**模型**：
- 测试分数 = g × 因子载荷 + 特定能力 + 噪声
- 高 g → 在所有领域都有更好的表现

我们用合成测试数据对此进行模拟。

In [ ]:
def generate_cognitive_test_data(n_subjects=200, n_tests=8, g_variance=0.7):
    """生成具有 g 因子结构的合成认知测试分数。
    
    参数：
        n_subjects：测试对象数量
        n_tests：不同认知测试的数量
        g_variance：g 因子解释的方差比例
        
    返回：
        scores：（n_subjects、n_tests）测试成绩
        g_factor：（n_subjects，）底层通用智能"""
    # 为每个受试者生成基础 g 因子（一般智力）
    g_factor = np.random.randn(n_subjects)
    
    # 测试载荷：每次测试取决于 g 的程度
    # 更高的负载=更多的g依赖性
    loadings = np.random.uniform(0.5, 0.9, n_tests)
    
    # 生成分数
    scores = np.zeros((n_subjects, n_tests))
    
    for i in range(n_tests):
        # 分数 = g 分量 + 特异性能力 + 噪声
        g_component = g_factor * loadings[i] * np.sqrt(g_variance)
        specific = np.random.randn(n_subjects) * np.sqrt(1 - g_variance)
        scores[:, i] = g_component + specific
    
    # 标准化为 0-100 范围
    scores = 50 + 15 * scores  # 平均值=50，SD=15（如智商）
    scores = np.clip(scores, 0, 100)
    
    return scores, g_factor, loadings

# 生成测试数据
test_names = ['Verbal', 'Spatial', 'Memory', 'Reasoning', 'Processing', 'Attention', 'Math', 'Pattern']
scores, g_factor, loadings = generate_cognitive_test_data(n_subjects=200, n_tests=len(test_names))

print("Generated cognitive test data:")
print(f"Subjects: {scores.shape[0]}")
print(f"Tests: {scores.shape[1]}")
print(f"\nMean scores per test:")
for i, name in enumerate(test_names):
    print(f"  {name:12s}: {scores[:, i].mean():.1f} ± {scores[:, i].std():.1f}")

In [ ]:
# 计算相关矩阵（显示正相关结构）
correlation_matrix = np.corrcoef(scores.T)

plt.figure(figsize=(10, 8))
plt.imshow(correlation_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(label='Correlation')
plt.xticks(range(len(test_names)), test_names, rotation=45)
plt.yticks(range(len(test_names)), test_names)
plt.title('Correlation Matrix of Cognitive Tests\n(Positive Manifold Shows g-factor)', fontsize=14)

# 添加相关值
for i in range(len(test_names)):
    for j in range(len(test_names)):
        plt.text(j, i, f'{correlation_matrix[i, j]:.2f}', 
                ha='center', va='center', color='white' if abs(correlation_matrix[i, j]) > 0.5 else 'black',
                fontsize=8)

plt.tight_layout()
plt.show()

print("\nKey observation: All tests show POSITIVE correlations")
print("This 'positive manifold' suggests a common underlying factor (g).")
print(f"Mean off-diagonal correlation: {np.mean(correlation_matrix[np.triu_indices_from(correlation_matrix, k=1)]):.3f}")

In [ ]:
# 使用主成分分析 (PCA) 提取 g 因子
def extract_g_factor(scores):
    """提取 g 因子作为第一主成分。
    
    第一主成分 捕获最大方差并表示
    所有测试的共同因素。"""
    # 将数据居中
    scores_centered = scores - scores.mean(axis=0)
    
    # 计算协方差矩阵
    cov_matrix = np.cov(scores_centered.T)
    
    # 特征分解
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
    
    # 按特征值排序（降序）
    idx = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    # 第一个分量是 g 因子
    g_extracted = scores_centered @ eigenvectors[:, 0]
    
    # 方差解释
    variance_explained = eigenvalues / eigenvalues.sum()
    
    return g_extracted, variance_explained, eigenvectors[:, 0]

g_extracted, var_explained, g_loadings = extract_g_factor(scores)

# 可视化解释方差
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 碎石图
axes[0].bar(range(1, len(var_explained) + 1), var_explained * 100, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained (%)')
axes[0].set_title(f'Scree Plot\nFirst PC (g-factor) explains {var_explained[0]*100:.1f}% of variance')
axes[0].grid(alpha=0.3)

# g 因子载荷
axes[1].barh(test_names, np.abs(g_loadings), color='coral', alpha=0.7)
axes[1].set_xlabel('Absolute Loading on g-factor')
axes[1].set_title('Test Loadings on g-factor\n(How much each test measures g)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\ng-factor (first PC) explains {var_explained[0]*100:.1f}% of total variance")
print(f"Correlation between true g and extracted g: {np.corrcoef(g_factor, g_extracted)[0, 1]:.3f}")

## 1.2 从心理测量到通用智能

**心理测量学中 g 因子的局限性**：
- 以人为中心（仅衡量类人智能）
- 取决于测试（结果因测试选择而异）
- 没有正式的数学定义

**Legg 与 Hutter 的通用智能**通过形式化定义智能来解决这些问题：

$$\Upsilon(\pi) = \sum_{\mu \in E} 2^{-K(\mu)} V_\mu^\pi$$

其中：
- $\pi$ 表示智能体
- $E$ 是所有可计算环境的集合
- $K(\mu)$ 表示环境 $\mu$ 的 Kolmogorov 复杂度
- $V_\mu^\pi$ 是环境 $\mu$ 中的预期奖励
- $2^{-K(\mu)}$ 使较简单的环境获得更高权重（Solomonoff 先验）

这个定义：
- **通用**（适用于任何智能体和任何环境）
- 是**形式化的**（精确的数学定义）
- 与**奥卡姆剃刀**保持一致（更简单的环境权重更大）
- **不可计算**（但可以近似！）

---

# 第 2 节：通用人工智能与 Solomonoff 归纳

## 2.1 Solomonoff 归纳

**问题**：给定一系列观察结果，预测下一个符号。

**Solomonoff 的解决方案**：考虑所有可计算的假设，并按其**Kolmogorov 复杂度**（生成它们的最短程序的长度）进行加权。

$$P(x_{1:n}) = \sum_{p: U(p) = x_{1:n}} 2^{-|p|}$$

其中：
- $U$ 是通用图灵机
- $p$ 是一个程序
- $|p|$ 是程序长度
- 较短的程序具有较高的先验概率

**关键属性**：
1. 通用：渐进地支配任何可计算的预测器
2. 最佳：比任何其他方法更快地收敛到真实分布
3. 无法计算：没有算法可以计算精确的 Solomonoff 概率

我们使用**简单的程序枚举**对此进行近似。

In [ ]:
class SimpleProgramEnumerator:
    """使用简单的程序枚举来对 Solomonoff 归纳的玩具级近似归纳法。
    
    我们枚举短程序（有限状态机）并对它们进行加权
    通过 2^(-length) 来近似 Solomonoff 先验。"""
    
    def __init__(self, alphabet_size=2, max_program_length=8):
        self.alphabet_size = alphabet_size
        self.max_length = max_program_length
        self.programs = []  # （程序、权重）元组列表
        
    def enumerate_programs(self):
        """将简单程序枚举为重复模式。
        
        程序被表示为重复的短序列。
        例如，[0, 1] 表示010101..."""
        programs = []
        
        # 枚举直到 max_length 的所有序列
        for length in range(1, self.max_length + 1):
            for pattern in itertools.product(range(self.alphabet_size), repeat=length):
                program = list(pattern)
                weight = 2.0 ** (-length)  # Solomonoff 先验
                programs.append((program, weight))
        
        # 标准化权重
        total_weight = sum(w for _, w in programs)
        programs = [(p, w / total_weight) for p, w in programs]
        
        self.programs = programs
        return len(programs)
    
    def generate_sequence(self, program, length):
        '通过重复程序模式生成序列。'
        seq = []
        for i in range(length):
            seq.append(program[i % len(program)])
        return np.array(seq)
    
    def predict_next(self, observed_sequence):
        """使用Solomonoff式加权投票来预测下一个符号。
        
        对于每个 program：
        1. 检查是否与观察到的序列一致
        2. 如果是，看看接下来的预测
        3. 通过程序的先验概率进行权重预测"""
        n = len(observed_sequence)
        
        # 累积加权预测
        predictions = np.zeros(self.alphabet_size)
        total_weight = 0.0
        
        for program, weight in self.programs:
            # 生成该程序将产生的内容
            generated = self.generate_sequence(program, n + 1)
            
            # 检查是否与观察结果一致
            if np.array_equal(generated[:n], observed_sequence):
                next_symbol = generated[n]
                predictions[next_symbol] += weight
                total_weight += weight
        
        if total_weight > 0:
            predictions /= total_weight
        else:
            # 如果没有一致的程序，则使用均匀分布
            predictions = np.ones(self.alphabet_size) / self.alphabet_size
        
        return predictions

# 创建预测器
predictor = SimpleProgramEnumerator(alphabet_size=2, max_program_length=6)
n_programs = predictor.enumerate_programs()

print(f"Enumerated {n_programs} programs")
print(f"\nExample programs (pattern, weight):")
for i in range(min(10, len(predictor.programs))):
    program, weight = predictor.programs[i]
    print(f"  {program} -> weight = {weight:.6f}")

In [ ]:
# 不同序列的测试
test_sequences = [
    ([0, 1, 0, 1, 0, 1], "Alternating"),
    ([1, 1, 1, 1, 1, 1], "Constant"),
    ([0, 1, 1, 0, 1, 1], "Pattern 011"),
    ([0, 0, 1, 0, 0, 1], "Pattern 001"),
]

print("Solomonoff Predictions:\n")
for seq, description in test_sequences:
    seq_array = np.array(seq)
    pred = predictor.predict_next(seq_array)
    
    print(f"{description:15s}: {seq}")
    print(f"  P(next=0) = {pred[0]:.4f}")
    print(f"  P(next=1) = {pred[1]:.4f}")
    print(f"  Prediction: {np.argmax(pred)}\n")

In [ ]:
# 可视化程序长度分布
program_lengths = [len(p) for p, _ in predictor.programs]
weights_by_length = defaultdict(float)

for program, weight in predictor.programs:
    weights_by_length[len(program)] += weight

lengths = sorted(weights_by_length.keys())
total_weights = [weights_by_length[l] for l in lengths]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 按程序长度划分的总权重
axes[0].bar(lengths, total_weights, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Program Length (bits)')
axes[0].set_ylabel('Total Prior Probability')
axes[0].set_title('Solomonoff Prior Distribution\n(Shorter programs weighted more)')
axes[0].grid(alpha=0.3)

# 按长度划分的程序数量
length_counts = [sum(1 for l in program_lengths if l == length) for length in lengths]
axes[1].bar(lengths, length_counts, color='coral', alpha=0.7)
axes[1].set_xlabel('Program Length (bits)')
axes[1].set_ylabel('Number of Programs')
axes[1].set_title('Program Count by Length\n(Exponential growth in hypotheses)')
axes[1].set_yscale('log')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insight: Solomonoff prior favors simplicity (Occam's Razor)")
print(f"Programs of length 1 have total weight: {weights_by_length[1]:.4f}")
print(f"Programs of length 6 have total weight: {weights_by_length[6]:.4f}")

## 2.2 序列预测性能

我们测试了Solomonoff 归纳与更简单的基线相比学习不同模式的效果。

In [ ]:
def test_sequence_prediction(true_pattern, seq_length=30, description=""):
    """随着我们观察更多序列，测试预测的准确性。
    
    比较对象：
    - Solomonoff 近似
    - 频率基线（预测迄今为止最常见的符号）
    - 随机基线"""
    # 生成真实序列
    true_seq = []
    for i in range(seq_length):
        true_seq.append(true_pattern[i % len(true_pattern)])
    true_seq = np.array(true_seq)
    
    solomonoff_correct = []
    frequency_correct = []
    
    # 看到一些符号后开始预测
    for t in range(3, seq_length):
        observed = true_seq[:t]
        true_next = true_seq[t]
        
        # Solomonoff预测
        sol_pred = predictor.predict_next(observed)
        sol_correct = (np.argmax(sol_pred) == true_next)
        solomonoff_correct.append(sol_correct)
        
        # 频率基线：预测最常见的符号
        freq_pred = 1 if np.sum(observed) > len(observed) / 2 else 0
        freq_correct = (freq_pred == true_next)
        frequency_correct.append(freq_correct)
    
    return solomonoff_correct, frequency_correct

# 测试不同的模式
patterns = [
    ([0, 1], "Alternating"),
    ([0, 0, 1], "Pattern 001"),
    ([1, 0, 1, 1], "Pattern 1011"),
]

fig, axes = plt.subplots(1, len(patterns), figsize=(16, 4))

for idx, (pattern, desc) in enumerate(patterns):
    sol_acc, freq_acc = test_sequence_prediction(pattern, seq_length=30, description=desc)
    
    # 计算累积精度
    sol_cumavg = np.cumsum(sol_acc) / np.arange(1, len(sol_acc) + 1)
    freq_cumavg = np.cumsum(freq_acc) / np.arange(1, len(freq_acc) + 1)
    
    axes[idx].plot(sol_cumavg, label='Solomonoff', linewidth=2, color='steelblue')
    axes[idx].plot(freq_cumavg, label='Frequency', linewidth=2, color='coral', linestyle='--')
    axes[idx].axhline(0.5, color='gray', linestyle=':', label='Random')
    axes[idx].set_xlabel('Observations')
    axes[idx].set_ylabel('Cumulative Accuracy')
    axes[idx].set_title(f'{desc}\nPattern: {"".join(map(str, pattern))}')
    axes[idx].legend()
    axes[idx].grid(alpha=0.3)
    axes[idx].set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

print("Solomonoff induction quickly identifies simple patterns!")
print("It outperforms frequency baseline by considering program simplicity.")

## 2.3 Kolmogorov复杂度近似

**Kolmogorov 复杂度** $K(x)$ 是输出 $x$ 的最短程序的长度。

虽然通常无法计算，但我们可以使用以下方法对其进行近似：
1. **压缩**：$K(x) \approx$ 压缩长度
2. **程序搜索**：寻找生成 $x$的最短程序

这里采用方法 2，也就是在枚举的程序中进行搜索。

In [ ]:
def estimate_kolmogorov_complexity(sequence):
    """
    通过寻找能够生成序列的最短程序来估计 K(sequence)。
    """
    n = len(sequence)
    min_length = float('inf')
    best_program = None
    
    for program, weight in predictor.programs:
        generated = predictor.generate_sequence(program, n)
        if np.array_equal(generated, sequence):
            if len(program) < min_length:
                min_length = len(program)
                best_program = program
    
    return min_length, best_program

# 测试具有不同复杂度的序列
test_sequences_k = [
    (np.array([0, 0, 0, 0, 0, 0]), "All zeros"),
    (np.array([0, 1, 0, 1, 0, 1]), "Alternating"),
    (np.array([0, 0, 1, 0, 0, 1]), "Pattern 001"),
    (np.array([0, 1, 1, 0, 1, 1]), "Pattern 011"),
    (np.array([1, 0, 0, 1, 1, 0]), "No simple pattern"),
]

print("Kolmogorov Complexity Estimates:\n")
complexities = []
labels = []

for seq, desc in test_sequences_k:
    k_est, program = estimate_kolmogorov_complexity(seq)
    complexities.append(k_est)
    labels.append(desc)
    
    print(f"{desc:20s}: K ≈ {k_est} bits")
    if program is not None:
        print(f"  Shortest program: {"".join(map(str, program))}")
        print(f"  Sequence: {"".join(map(str, seq.tolist()))}\n")
    else:
        print(f"  No program found (complexity > {predictor.max_length})\n")

In [ ]:
# 可视化复杂性
plt.figure(figsize=(10, 6))
colors = ['green', 'blue', 'orange', 'orange', 'red']
bars = plt.barh(labels, complexities, color=colors, alpha=0.7)
plt.xlabel('Estimated Kolmogorov Complexity (bits)')
plt.title('Sequence Complexity Estimates\n(Shorter = simpler)', fontsize=14)
plt.grid(axis='x', alpha=0.3)

# 添加值
for i, (complexity, label) in enumerate(zip(complexities, labels)):
    plt.text(complexity + 0.1, i, f'{complexity}', va='center')

plt.tight_layout()
plt.show()

print("\nSimpler patterns have lower Kolmogorov complexity!")
print("This formalizes Occam's Razor: prefer simpler explanations.")

---

# 第 3 节：AIXI 智能体与环境模型

## 3.1 AIXI 智能体

**AIXI** 是理论上最优的强化学习智能体（Hutter，2005）。

**动作选择**：

$$a_t^* = \arg\max_{a_t} \sum_{o_t, r_t} \max_{a_{t+1}} \sum_{o_{t+1}, r_{t+1}} \cdots \max_{a_m} \sum_{o_m, r_m} [r_t + \cdots + r_m] \cdot P(o_t r_t \cdots o_m r_m | a_t \cdots a_m)$$

其中：
$$P(\text{observations} | \text{actions}) = \sum_{\mu \in E} 2^{-K(\mu)} P_\mu(\text{observations} | \text{actions})$$

AIXI：
1. 考虑按Kolmogorov复杂度加权的所有可能环境
2. 使用极大极小搜索对动作序列进行最优规划
3. 不可计算（需要无限计算）

**蒙特卡罗 AIXI (MC-AIXI)** 使用以下方法近似计算：
- 对一小组环境假设进行采样
- 蒙特卡罗树搜索规划
- 有限的前瞻范围

我们在玩具网格世界环境中实现了简化的 MC-AIXI。

In [ ]:
class ToyGridWorld:
    """简单的 5x5 网格世界环境。
    
    智能体从 (0, 0) 开始，必须在 (4, 4) 处到达目标。
    动作：上、下、左、右
    奖励：达到目标+10，每步-1"""
    
    def __init__(self, size=5):
        self.size = size
        self.reset()
        
    def reset(self):
        self.agent_pos = [0, 0]
        self.goal_pos = [self.size - 1, self.size - 1]
        self.done = False
        self.total_reward = 0
        return self.get_observation()
    
    def get_observation(self):
        '返回智能体位置作为观察。'
        return tuple(self.agent_pos)
    
    def step(self, action):
        """执行行动。
        动作：0=上，1=下，2=左，3=右"""
        if self.done:
            return self.get_observation(), 0, True
        
        # 执行动作
        if action == 0:  # 向上
            self.agent_pos[0] = max(0, self.agent_pos[0] - 1)
        elif action == 1:  # 向下
            self.agent_pos[0] = min(self.size - 1, self.agent_pos[0] + 1)
        elif action == 2:  # 向左
            self.agent_pos[1] = max(0, self.agent_pos[1] - 1)
        elif action == 3:  # 向右
            self.agent_pos[1] = min(self.size - 1, self.agent_pos[1] + 1)
        
        # 检查目标是否达到
        reward = -1  # 步罚分
        if self.agent_pos == self.goal_pos:
            reward = 10
            self.done = True
        
        self.total_reward += reward
        return self.get_observation(), reward, self.done
    
    def copy(self):
        '创建一个副本以进行模拟。'
        new_env = ToyGridWorld(self.size)
        new_env.agent_pos = self.agent_pos.copy()
        new_env.done = self.done
        new_env.total_reward = self.total_reward
        return new_env

# 测试环境
env = ToyGridWorld(size=5)
obs = env.reset()

print("Toy Grid World Environment")
print(f"Size: {env.size}x{env.size}")
print(f"Start: {env.agent_pos}")
print(f"Goal: {env.goal_pos}")
print(f"Actions: 0=up, 1=down, 2=left, 3=right")
print(f"\nInitial observation: {obs}")

# 尝试随机动作
print("\nRandom episode:")
total_reward = 0
for step in range(20):
    action = np.random.randint(0, 4)
    obs, reward, done = env.step(action)
    total_reward += reward
    print(f"  Step {step}: action={action}, pos={obs}, reward={reward:.1f}")
    if done:
        print(f"  Goal reached! Total reward: {total_reward}")
        break

In [ ]:
class MCTreeNode:
    '蒙特卡罗树搜索中的节点。'
    
    def __init__(self, state, parent=None, action=None):
        self.state = state
        self.parent = parent
        self.action = action
        self.children = []
        self.visits = 0
        self.value = 0.0
    
    def is_fully_expanded(self, n_actions):
        return len(self.children) == n_actions
    
    def best_child(self, exploration_weight=1.0):
        '使用 UCB1 选择子节点。'
        choices_weights = []
        for child in self.children:
            if child.visits == 0:
                weight = float('inf')
            else:
                exploit = child.value / child.visits
                explore = exploration_weight * np.sqrt(np.log(self.visits) / child.visits)
                weight = exploit + explore
            choices_weights.append(weight)
        return self.children[np.argmax(choices_weights)]

class SimpleMCAIXI:
    """简化的蒙特卡罗 AIXI 智能体。
    
    使用蒙特卡罗树搜索 (MCTS) 来规划操作。"""
    
    def __init__(self, n_actions=4, n_simulations=100, horizon=5):
        self.n_actions = n_actions
        self.n_simulations = n_simulations
        self.horizon = horizon
    
    def select_action(self, env):
        """使用 MCTS 选择操作。
        
        1. 选择：使用 UCB1 遍历搜索树
        2. 扩展：添加新的子节点
        3. 模拟：执行随机策略的 rollout
        4. 反向传播：更新节点价值"""
        root = MCTreeNode(env.copy())
        
        for _ in range(self.n_simulations):
            node = root
            sim_env = env.copy()
            
            # 选择
            while node.is_fully_expanded(self.n_actions) and len(node.children) > 0:
                node = node.best_child()
                sim_env.step(node.action)
            
            # 扩展
            if not node.is_fully_expanded(self.n_actions) and not sim_env.done:
                action = len(node.children)  # 尝试下一个未尝试过的操作
                new_env = sim_env.copy()
                new_env.step(action)
                child = MCTreeNode(new_env, parent=node, action=action)
                node.children.append(child)
                node = child
                sim_env = new_env
            
            # 模拟（推出）
            rollout_reward = 0
            for _ in range(self.horizon):
                if sim_env.done:
                    break
                action = np.random.randint(0, self.n_actions)
                _, reward, _ = sim_env.step(action)
                rollout_reward += reward
            
            # 反向传播
            while node is not None:
                node.visits += 1
                node.value += rollout_reward
                node = node.parent
        
        # 选择最佳行动
        if len(root.children) == 0:
            return np.random.randint(0, self.n_actions)
        
        best_child = max(root.children, key=lambda c: c.visits)
        return best_child.action

# 测试 MC-AIXI 智能体
print("Testing MC-AIXI agent...\n")

agent = SimpleMCAIXI(n_actions=4, n_simulations=50, horizon=10)
env = ToyGridWorld(size=5)
obs = env.reset()

episode_reward = 0
for step in range(20):
    action = agent.select_action(env)
    obs, reward, done = env.step(action)
    episode_reward += reward
    
    action_names = ['UP', 'DOWN', 'LEFT', 'RIGHT']
    print(f"Step {step}: {action_names[action]:5s} -> pos={obs}, reward={reward:+.1f}")
    
    if done:
        print(f"\nGoal reached in {step + 1} steps!")
        print(f"Total reward: {episode_reward}")
        break

if not done:
    print(f"\nDid not reach goal. Total reward: {episode_reward}")

## 3.2 比较不同智能体

我们来比较一下不同的智能体：
1. **随机**：均匀随机选择动作
2. **贪婪**：朝目标移动（曼哈顿距离）
3. **MC-AIXI**：使用 MCTS 规划

In [ ]:
def random_agent(env):
    '随机动作选择。'
    return np.random.randint(0, 4)

def greedy_agent(env):
    """贪心智能体：向目标前进。
    
    计算曼哈顿距离并选择减少该距离的操作。"""
    agent_pos = env.agent_pos
    goal_pos = env.goal_pos
    
    # 垂直运动
    if agent_pos[0] < goal_pos[0]:
        return 1  # 向下
    elif agent_pos[0] > goal_pos[0]:
        return 0  # 向上
    # 水平运动
    elif agent_pos[1] < goal_pos[1]:
        return 3  # 向右
    elif agent_pos[1] > goal_pos[1]:
        return 2  # 向左
    else:
        return np.random.randint(0, 4)

def evaluate_agent(agent_fn, n_episodes=20, max_steps=30):
    '在多个情节中评估智能体。'
    total_rewards = []
    steps_to_goal = []
    
    for _ in range(n_episodes):
        env = ToyGridWorld(size=5)
        env.reset()
        
        episode_reward = 0
        for step in range(max_steps):
            action = agent_fn(env)
            obs, reward, done = env.step(action)
            episode_reward += reward
            
            if done:
                steps_to_goal.append(step + 1)
                break
        
        total_rewards.append(episode_reward)
    
    return total_rewards, steps_to_goal

# 评估不同智能体
print("Evaluating agents over 20 episodes...\n")

mc_aixi_agent = SimpleMCAIXI(n_actions=4, n_simulations=30, horizon=10)

random_rewards, random_steps = evaluate_agent(random_agent, n_episodes=20)
greedy_rewards, greedy_steps = evaluate_agent(greedy_agent, n_episodes=20)
aixi_rewards, aixi_steps = evaluate_agent(lambda env: mc_aixi_agent.select_action(env), n_episodes=20)

# 结果
print("Results (mean ± std):")
print(f"\nRandom Agent:")
print(f"  Reward: {np.mean(random_rewards):.1f} ± {np.std(random_rewards):.1f}")
print(f"  Success rate: {len(random_steps)/20*100:.0f}%")
if len(random_steps) > 0:
    print(f"  Steps to goal: {np.mean(random_steps):.1f} ± {np.std(random_steps):.1f}")

print(f"\nGreedy Agent:")
print(f"  Reward: {np.mean(greedy_rewards):.1f} ± {np.std(greedy_rewards):.1f}")
print(f"  Success rate: {len(greedy_steps)/20*100:.0f}%")
if len(greedy_steps) > 0:
    print(f"  Steps to goal: {np.mean(greedy_steps):.1f} ± {np.std(greedy_steps):.1f}")

print(f"\nMC-AIXI Agent:")
print(f"  Reward: {np.mean(aixi_rewards):.1f} ± {np.std(aixi_rewards):.1f}")
print(f"  Success rate: {len(aixi_steps)/20*100:.0f}%")
if len(aixi_steps) > 0:
    print(f"  Steps to goal: {np.mean(aixi_steps):.1f} ± {np.std(aixi_steps):.1f}")

In [ ]:
# 可视化表现
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 奖励比较
agent_names = ['Random', 'Greedy', 'MC-AIXI']
mean_rewards = [np.mean(random_rewards), np.mean(greedy_rewards), np.mean(aixi_rewards)]
std_rewards = [np.std(random_rewards), np.std(greedy_rewards), np.std(aixi_rewards)]

axes[0].bar(agent_names, mean_rewards, yerr=std_rewards, capsize=5, 
           color=['gray', 'steelblue', 'coral'], alpha=0.7)
axes[0].set_ylabel('Total Reward')
axes[0].set_title('Agent Performance\n(Higher is better)')
axes[0].grid(axis='y', alpha=0.3)

# 目标比较的步骤
steps_data = [random_steps, greedy_steps, aixi_steps]
axes[1].boxplot(steps_data, labels=agent_names)
axes[1].set_ylabel('Steps to Goal')
axes[1].set_title('Efficiency\n(Lower is better)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nMC-AIXI combines planning and lookahead to achieve better performance!")
print("Greedy agent is fast but suboptimal in complex environments.")
print("Random agent shows the baseline for comparison.")

---

# 第 4 节：通用智能度量

## 4.1 形式定义

**Legg 与 Hutter 的通用智能度量**：

$$\Upsilon(\pi) = \sum_{\mu \in E} 2^{-K(\mu)} V_\mu^\pi$$

其中：
- $\pi$ 表示智能体策略
- $E$ 是所有可计算环境的空间
- $K(\mu)$ 表示环境 $\mu$ 的 Kolmogorov 复杂度
- $V_\mu^\pi$ 表示在环境 $\mu$ 中的期望折扣回报：

$$V_\mu^\pi = \mathbb{E}_{\pi, \mu}\left[\sum_{t=1}^\infty \gamma^t r_t\right]$$

**关键属性**：
1. **通用**：考虑所有可能的可计算环境
2. **以简单性为权重**：更简单的环境更重要（Solomonoff 先验）
3. **基于表现**：根据获得的奖励来衡量
4. **不可计算**：但可以用有限环境集进行近似

我们使用一小套玩具环境来近似 $\Upsilon(\pi)$。

In [ ]:
class EnvironmentSuite:
    '具有估计 Kolmogorov 复杂性的玩具环境集合。'
    
    def __init__(self):
        self.environments = self.create_environments()
    
    def create_environments(self):
        """创建具有不同复杂性的环境。
        
        每个环境返回：
        - env_fn：创建环境实例的函数
        - complexity：估计的 K(μ)（以位为单位）
        - description：人类可读的描述"""
        envs = []
        
        # Env 1：持续奖励（最简单）
        def constant_reward_env():
            class ConstantEnv:
                def __init__(self):
                    self.done = False
                def reset(self):
                    self.done = False
                    return 0
                def step(self, action):
                    return 0, 1.0, False  # 总是奖励+1
                def copy(self):
                    new = ConstantEnv()
                    new.done = self.done
                    return new
            return ConstantEnv()
        
        envs.append({
            'env_fn': constant_reward_env,
            'complexity': 2,  # 很简单：总是返回 1
            'description': 'Constant reward (+1)'
        })
        
        # Env 2：二元选择（左=0，右=1）
        def binary_choice_env():
            class BinaryEnv:
                def __init__(self):
                    self.done = False
                def reset(self):
                    self.done = False
                    return 0
                def step(self, action):
                    # 动作 3（右）给出 +1，其他给出 0
                    reward = 1.0 if action == 3 else 0.0
                    return 0, reward, False
                def copy(self):
                    return BinaryEnv()
            return BinaryEnv()
        
        envs.append({
            'env_fn': binary_choice_env,
            'complexity': 4,  # 需要编码：if action==3 then 1 else 0
            'description': 'Binary choice (right=+1)'
        })
        
        # Env 3：网格世界（更复杂）
        def grid_world_env():
            return ToyGridWorld(size=5)
        
        envs.append({
            'env_fn': grid_world_env,
            'complexity': 8,  # 需要编码：网格、运动动态、目标
            'description': 'Grid world navigation'
        })
        
        return envs
    
    def compute_universal_intelligence(self, agent_fn, n_episodes=10, max_steps=20, gamma=0.95):
        """计算 Υ(π) 的近似值。
        
        Υ(π) ≈ Σ_μ 2^(-K(μ)) * V_μ^π"""
        intelligence = 0.0
        environment_values = []
        
        for env_spec in self.environments:
            # 获取环境参数
            env_fn = env_spec['env_fn']
            complexity = env_spec['complexity']
            
            # 计算环境权重（Solomonoff先验）
            weight = 2.0 ** (-complexity)
            
            # 估计V_μ^π（预期折扣回报）
            returns = []
            for _ in range(n_episodes):
                env = env_fn()
                env.reset()
                
                discounted_return = 0.0
                discount = 1.0
                
                for step in range(max_steps):
                    action = agent_fn(env)
                    _, reward, done = env.step(action)
                    discounted_return += discount * reward
                    discount *= gamma
                    
                    if done:
                        break
                
                returns.append(discounted_return)
            
            value = np.mean(returns)
            environment_values.append({
                'description': env_spec['description'],
                'complexity': complexity,
                'weight': weight,
                'value': value,
                'weighted_value': weight * value
            })
            
            # 添加到智力测量
            intelligence += weight * value
        
        # 按总权重归一化
        total_weight = sum(2.0 ** (-env['complexity']) for env in self.environments)
        intelligence /= total_weight
        
        return intelligence, environment_values

# 创建环境套件
suite = EnvironmentSuite()

print("Environment Suite:")
print()
for i, env_spec in enumerate(suite.environments, 1):
    weight = 2.0 ** (-env_spec['complexity'])
    print(f"{i}. {env_spec['description']}")
    print(f"   K(μ) ≈ {env_spec['complexity']} bits")
    print(f"   Weight = 2^(-{env_spec['complexity']}) = {weight:.4f}")
    print()

In [ ]:
# 不同智能体的计算智能
print("Computing Universal Intelligence Υ(π) for each agent...\n")

mc_aixi_light = SimpleMCAIXI(n_actions=4, n_simulations=20, horizon=8)

# 随机智能体
upsilon_random, env_values_random = suite.compute_universal_intelligence(
    random_agent, n_episodes=10, max_steps=15
)

# 贪心智能体
upsilon_greedy, env_values_greedy = suite.compute_universal_intelligence(
    greedy_agent, n_episodes=10, max_steps=15
)

# MC-AIXI智能体
upsilon_aixi, env_values_aixi = suite.compute_universal_intelligence(
    lambda env: mc_aixi_light.select_action(env), n_episodes=10, max_steps=15
)

print("Universal Intelligence Υ(π):")
print(f"  Random:  {upsilon_random:.3f}")
print(f"  Greedy:  {upsilon_greedy:.3f}")
print(f"  MC-AIXI: {upsilon_aixi:.3f}")
print()

# 按环境显示细分
print("Breakdown by environment (MC-AIXI):")
for ev in env_values_aixi:
    print(f"  {ev['description']:30s}: V={ev['value']:6.2f}, weight={ev['weight']:.4f}, contribution={ev['weighted_value']:.3f}")

In [ ]:
# 可视化智力比较
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 整体智能
agents = ['Random', 'Greedy', 'MC-AIXI']
intelligence_scores = [upsilon_random, upsilon_greedy, upsilon_aixi]

axes[0].bar(agents, intelligence_scores, color=['gray', 'steelblue', 'coral'], alpha=0.7)
axes[0].set_ylabel('Υ(π)')
axes[0].set_title('Universal Intelligence Measure\n(Higher = more intelligent)', fontsize=12)
axes[0].grid(axis='y', alpha=0.3)

# 在条形图上添加值
for i, score in enumerate(intelligence_scores):
    axes[0].text(i, score + 0.05, f'{score:.3f}', ha='center', fontsize=11, fontweight='bold')

# 不同环境下的性能
env_names = [ev['description'] for ev in env_values_aixi]
random_values = [ev['value'] for ev in env_values_random]
greedy_values = [ev['value'] for ev in env_values_greedy]
aixi_values = [ev['value'] for ev in env_values_aixi]

x = np.arange(len(env_names))
width = 0.25

axes[1].bar(x - width, random_values, width, label='Random', color='gray', alpha=0.7)
axes[1].bar(x, greedy_values, width, label='Greedy', color='steelblue', alpha=0.7)
axes[1].bar(x + width, aixi_values, width, label='MC-AIXI', color='coral', alpha=0.7)

axes[1].set_ylabel('Expected Return V_μ^π')
axes[1].set_title('Performance by Environment', fontsize=12)
axes[1].set_xticks(x)
axes[1].set_xticklabels(env_names, rotation=15, ha='right')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey insight: MC-AIXI achieves higher Υ(π) through better performance")
print("across diverse environments, especially complex ones!")

## 4.2 智能与环境复杂性

通用智能度量更重视简单的环境。这意味着：
- 简单环境下的性能更重要
- 但智能体仍然必须应对多样化的挑战

下面通过可视化观察环境复杂性如何影响智能度量。

In [ ]:
# 按复杂性分析贡献
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Solomonoff按复杂性加权
complexities = [ev['complexity'] for ev in env_values_aixi]
weights = [ev['weight'] for ev in env_values_aixi]
contributions_aixi = [ev['weighted_value'] for ev in env_values_aixi]

axes[0].bar(env_names, weights, color='purple', alpha=0.6)
axes[0].set_ylabel('Solomonoff Weight (2^-K)')
axes[0].set_title('Environment Weighting\n(Simpler = higher weight)', fontsize=12)
axes[0].set_xticks(range(len(env_names)))
axes[0].set_xticklabels(env_names, rotation=15, ha='right')
axes[0].grid(axis='y', alpha=0.3)

# 添加复杂性标签
for i, (w, k) in enumerate(zip(weights, complexities)):
    axes[0].text(i, w + 0.01, f'K={k}', ha='center', fontsize=9)

# 对 Υ(π) 的贡献
axes[1].bar(env_names, contributions_aixi, color='coral', alpha=0.7)
axes[1].set_ylabel('Contribution to Υ(π)')
axes[1].set_title('Environment Contributions\n(weight × value)', fontsize=12)
axes[1].set_xticks(range(len(env_names)))
axes[1].set_xticklabels(env_names, rotation=15, ha='right')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("Simpler environments (lower K) contribute more to intelligence measure.")
print("This embodies Occam's razor: simple patterns matter most.")

---

# 第 5 节：近似方法与计算限制

## 5.1 为什么 AIXI 不可计算

**理论 AIXI**：
- 考虑所有可计算环境
- 使用精确的Kolmogorov复杂度（无法计算）
- 执行无限前瞻规划

**实用近似方法**：
1. **MC-AIXI**：对环境假设进行采样，使用 MCTS
2. **限时 AIXI**：限制计算时间
3. **受限假设空间**：仅考虑简单的环境模型

我们探索**计算时间与性能之间的权衡**。

In [ ]:
# 比较不同计算预算的 MC-AIXI
def evaluate_with_budget(n_simulations_list, n_episodes=10):
    '使用不同的计算预算评估 MC-AIXI。'
    results = []
    
    for n_sims in n_simulations_list:
        agent = SimpleMCAIXI(n_actions=4, n_simulations=n_sims, horizon=8)
        
        # 在网格世界上进行测试
        rewards = []
        for _ in range(n_episodes):
            env = ToyGridWorld(size=5)
            env.reset()
            
            episode_reward = 0
            for step in range(20):
                action = agent.select_action(env)
                _, reward, done = env.step(action)
                episode_reward += reward
                if done:
                    break
            
            rewards.append(episode_reward)
        
        results.append({
            'simulations': n_sims,
            'mean_reward': np.mean(rewards),
            'std_reward': np.std(rewards)
        })
    
    return results

# 测试不同的预算
print("Testing computation budget effect...\n")
budgets = [5, 10, 20, 50, 100]
budget_results = evaluate_with_budget(budgets, n_episodes=10)

print("Results:")
print(f"{'Simulations':<12} {'Mean Reward':<15} {'Std Reward':<15}")
print("-" * 45)
for res in budget_results:
    print(f"{res['simulations']:<12} {res['mean_reward']:<15.2f} {res['std_reward']:<15.2f}")

In [ ]:
# 可视化性能与计算
simulations = [res['simulations'] for res in budget_results]
mean_rewards = [res['mean_reward'] for res in budget_results]
std_rewards = [res['std_reward'] for res in budget_results]

plt.figure(figsize=(10, 6))
plt.errorbar(simulations, mean_rewards, yerr=std_rewards, 
             marker='o', markersize=8, capsize=5, linewidth=2,
             color='steelblue', label='MC-AIXI')
plt.axhline(np.mean(greedy_rewards), color='coral', linestyle='--', 
           linewidth=2, label='Greedy (no planning)')
plt.xlabel('MCTS Simulations (computation budget)', fontsize=12)
plt.ylabel('Mean Episode Reward', fontsize=12)
plt.title('Performance vs Computation Budget\n(Diminishing returns)', fontsize=14)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nKey observation: Performance improves with computation but plateaus.")
print("This shows the tradeoff between optimality and computability.")

## 5.2 近似质量

不同的近似会做出不同的权衡：

| 方法 | 计算量 | 最优性 | 实用吗？ |
|--------|-------------|------------|------------|
| 理论 AIXI | 无限 | 完美 | 否 |
| MC-AIXI（大预算） | 非常高 | 较好 | 有时 |
| MC-AIXI（小预算） | 中等 | 尚可 | 是 |
| 贪心/启发式方法 | 低 | 较差 | 是 |

这说明了以下之间的**根本矛盾**：
- **通用性**：处理所有环境
- **最优性**：做出最佳决策
- **可计算性**：在有限时间内运行

---

# 第 6 节：通往超级智能的路径

## 6.1 递归自我改进

实现超级智能的关键途径是**递归自我改进**：
1. 系统提升自身智能
2. 更智能的系统带来更好的改进
3. 进程加速（智能爆炸）

我们用一个玩具模型来模拟这一点，代理可以：
- 提高规划深度
- 增加 MCTS 模拟预算
- 学习环境模型

In [ ]:
class SelfImprovingAgent:
    """能够提升自身能力的智能体。
    
    改进机制：
    - 增加 MCTS模拟预算
    - 增加规划范围"""
    
    def __init__(self, initial_simulations=10, initial_horizon=5):
        self.simulations = initial_simulations
        self.horizon = initial_horizon
        self.improvement_history = [{
            'step': 0,
            'simulations': initial_simulations,
            'horizon': initial_horizon,
            'intelligence': 0
        }]
    
    def select_action(self, env):
        '使用当前能力选择动作。'
        agent = SimpleMCAIXI(
            n_actions=4,
            n_simulations=self.simulations,
            horizon=self.horizon
        )
        return agent.select_action(env)
    
    def self_improve(self, performance_feedback):
        """根据表现提升能力。
        
        更好的性能 → 更多的资源用于改进。
        这创建了正反馈循环。"""
        # 改进率与当前智力成正比
        improvement_factor = 1.0 + (performance_feedback / 10.0)
        
        # 增加计算预算
        self.simulations = int(self.simulations * improvement_factor)
        self.simulations = min(self.simulations, 200)  # 设置实用范围内的上限
        
        # 增加规划深度
        if np.random.rand() < 0.3:  # 偶尔增加规划范围
            self.horizon = min(self.horizon + 1, 15)
        
        self.improvement_history.append({
            'step': len(self.improvement_history),
            'simulations': self.simulations,
            'horizon': self.horizon,
            'intelligence': performance_feedback
        })

# 模拟自我提升过程
print("Simulating recursive self-improvement...\n")

agent = SelfImprovingAgent(initial_simulations=5, initial_horizon=3)

n_improvement_cycles = 8
for cycle in range(n_improvement_cycles):
    # 评估当前绩效
    rewards = []
    for _ in range(5):  # 每次评估 5 集
        env = ToyGridWorld(size=5)
        env.reset()
        
        episode_reward = 0
        for step in range(20):
            action = agent.select_action(env)
            _, reward, done = env.step(action)
            episode_reward += reward
            if done:
                break
        rewards.append(episode_reward)
    
    performance = np.mean(rewards)
    
    print(f"Cycle {cycle}:")
    print(f"  Simulations: {agent.simulations:4d}  Horizon: {agent.horizon:2d}")
    print(f"  Performance: {performance:6.2f}\n")
    
    # 自我提升
    agent.self_improve(performance)

In [ ]:
# 可视化智力爆炸
history = agent.improvement_history
steps = [h['step'] for h in history]
simulations_hist = [h['simulations'] for h in history]
horizon_hist = [h['horizon'] for h in history]
intelligence_hist = [h['intelligence'] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 能力成长
ax1 = axes[0]
ax1.plot(steps, simulations_hist, marker='o', linewidth=2, 
        color='steelblue', label='MCTS Simulations')
ax1.set_xlabel('Improvement Cycle', fontsize=12)
ax1.set_ylabel('MCTS Simulations', color='steelblue', fontsize=12)
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.grid(alpha=0.3)

ax1_twin = ax1.twinx()
ax1_twin.plot(steps, horizon_hist, marker='s', linewidth=2, 
             color='coral', label='Planning Horizon')
ax1_twin.set_ylabel('Planning Horizon', color='coral', fontsize=12)
ax1_twin.tick_params(axis='y', labelcolor='coral')

axes[0].set_title('Capability Growth\n(Recursive Self-Improvement)', fontsize=13)

# 智力增长（指数型）
axes[1].plot(steps, intelligence_hist, marker='o', linewidth=2.5, 
            color='purple', label='Performance')
axes[1].set_xlabel('Improvement Cycle', fontsize=12)
axes[1].set_ylabel('Performance (Intelligence)', fontsize=12)
axes[1].set_title('Intelligence Explosion\n(Accelerating improvement)', fontsize=13)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey insight: Self-improvement creates positive feedback loop.")
print("Better performance → more resources → better improvements → faster growth.")
print("This is the theoretical basis for 'intelligence explosion'.")

## 6.2 智能爆炸动力学

**影响起飞速度的因素**：

1. **优化能力**：系统自我改进的程度如何？
2. **改进阻力**：改进有多难？
3. **收益递减**：随着时间的推移，改进会变得越来越困难吗？

**起飞场景**：
- **缓慢起飞**：线性或亚线性增长（几十年到超级智能）
- **快速起飞**：指数增长（数年内达到超级智能）
- **硬起飞**：超指数增长（几天/几周达到超级智能）

我们对不同的增长曲线进行建模。

In [ ]:
def simulate_takeoff(growth_model, n_steps=50):
    """模拟不同模型下的智力增长。
    
    growth_model：函数（current_intelligence，步骤）-> new_intelligence"""
    intelligence = [1.0]  # 从人类基线水平 (1.0) 开始
    
    for step in range(1, n_steps):
        new_int = growth_model(intelligence[-1], step)
        intelligence.append(new_int)
    
    return intelligence

# 不同的增长模式
def linear_growth(current, step):
    '慢速起飞: I(t) = I(0) + kt'
    return 1.0 + 0.1 * step

def exponential_growth(current, step):
    '快速起飞: I(t) = I(0) * e^(kt)'
    return current * 1.15

def superexponential_growth(current, step):
    '硬起飞：I(t+1) = I(t)^k（递归改进）'
    growth_rate = 1.05 + (current - 1.0) * 0.01  # 加速
    return current * growth_rate

def diminishing_returns(current, step):
    '带饱和上限的慢速起飞：渐近极限'
    limit = 10.0
    rate = 0.15
    return current + rate * (limit - current)

# 模拟场景
linear_curve = simulate_takeoff(linear_growth, n_steps=50)
exponential_curve = simulate_takeoff(exponential_growth, n_steps=50)
superexp_curve = simulate_takeoff(superexponential_growth, n_steps=50)
diminishing_curve = simulate_takeoff(diminishing_returns, n_steps=50)

# 可视化
plt.figure(figsize=(12, 7))

steps = range(len(linear_curve))
plt.plot(steps, linear_curve, linewidth=2.5, label='Linear (Slow Takeoff)', color='blue')
plt.plot(steps, exponential_curve, linewidth=2.5, label='Exponential (Fast Takeoff)', color='orange')
plt.plot(steps, superexp_curve, linewidth=2.5, label='Super-exponential (Hard Takeoff)', color='red')
plt.plot(steps, diminishing_curve, linewidth=2.5, label='Diminishing Returns', color='green', linestyle='--')

plt.axhline(1.0, color='gray', linestyle=':', linewidth=1.5, label='Human-level')
plt.xlabel('Time Steps', fontsize=13)
plt.ylabel('Intelligence (human-level = 1.0)', fontsize=13)
plt.title('Intelligence Explosion Scenarios\n(Different growth dynamics)', fontsize=15)
plt.legend(fontsize=11, loc='upper left')
plt.grid(alpha=0.3)
plt.yscale('log')
plt.ylim([0.8, 1000])
plt.tight_layout()
plt.show()

print("Intelligence growth depends critically on feedback dynamics!")
print("\nAt step 50:")
print(f"  Linear:          {linear_curve[-1]:8.1f}x human-level")
print(f"  Exponential:     {exponential_curve[-1]:8.1f}x human-level")
print(f"  Super-exp:       {superexp_curve[-1]:8.1f}x human-level")
print(f"  Diminishing:     {diminishing_curve[-1]:8.1f}x human-level")

## 6.3 影响与风险

**通用智能理论带来的主要认识**：

1. **智力是可测量的**：Υ(π) 提供了正式定义
2. **AIXI 是最优的**：但无法计算（基本限制）
3. **存在近似值**：MC-AIXI 和变体在实践中有效
4. **自我提升是可能的**：导致潜在的智力爆炸

**开放式问题**：
- 真正的人工智能系统的改进速度有多快？
- 智力的基本限制是什么？
- 我们如何在递归自我改进过程中保持控制？
- 我们能让超级智能系统与人类价值观保持一致吗？

**与安全的联系**：
- AIXI 没有内在价值（仅最大化奖励）
- 奖励函数的规范至关重要
- 超级智能将非常有能力，但不一定与人类目标一致
- 在智能爆炸之前需要稳健的价值对齐

---

# 总结与要点

## 已实现的核心概念

1. **心理测量智能（g 因子）**
   - 模拟认知测试数据
   - 使用 PCA提取一般智力因子
   - 在相关矩阵中展示相关矩阵中的正相关结构

2. **Solomonoff 归纳**
   - 通过程序枚举近似通用先验
   - 演示了以简单性为加权的序列预测
   - 估计序列的 Kolmogorov 复杂度

3. **AIXI 智能体**
   - 使用 MCTS 实现 MC-AIXI
   - 在玩具网格世界环境中进行测试
   - 与随机基线和贪婪基线相比

4. **通用智能度量（Υ）**
   - 创建具有不同复杂性的环境套件
   - 计算不同智能体的 Υ(π)
   - 说明智能来自在多种环境中的综合表现

5. **计算限制**
   - 探索计算性能权衡
   - 随着预算的增加，收益递减
   - 说明完美 AIXI 的不可计算性

6. **递归自我改进**
   - 模拟智能体提高自身能力
   - 建模不同的智能爆炸场景
   - 讨论超级智能的影响

## 哲学意义

- **智能是可以正式定义的**（不仅仅是直觉）
- **简单性很重要**（Solomonoff先验 = 奥卡姆剃刀）
- **最优性是无法计算的**（基本限制）
- **自我改进创造反馈循环**（可能出现智力爆炸）

## 实践启示

- 真正的人工智能系统使用理论理想的**近似值**
- **计算预算**会显著影响性能
- **环境多样性**考验真正的智力
- 在实现超级智能之前需要完成**价值对齐**

---

**进一步阅读**：
- Legg, S. (2008). *Machine Super Intelligence*. PhD Thesis.
- Hutter, M. (2005). *Universal Artificial Intelligence*.
- Solomonoff, R. (1964). *A Formal Theory of Inductive Inference*.
- Bostrom, N. (2014). *Superintelligence: Paths, Dangers, Strategies*.

**相关论文**：
- 论文 23：MDL（最小描述长度）
- 论文 25：Kolmogorov复杂度
- 论文 8：DQN（实用深度强化学习）